# Calibration and Uncertainty

## Scientific objective
Compare uncalibrated, Platt, isotonic, and temperature scaling on validation data; form endpoint bundles with model-family disagreement uncertainty.

## Inputs
- Endpoint QSAR estimators
- Validation/test scaffold partitions

## Expected outputs
- `models/calibrated/*.joblib`
- `results/calibration/calibrated_model_summary.csv`
- reliability diagrams

## Dependencies
scikit-learn, SciPy, joblib

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
Calibration is selected per endpoint using validation Brier score, with PR-AUC as a secondary criterion. Test labels are used only once for reporting.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Model-family disagreement is a practical uncertainty proxy, not a proof of epistemic calibration. Deep ensembles and conformal intervals should be evaluated in the full profile.

## Next notebook
[17_applicability_domain_and_ood.ipynb](./17_applicability_domain_and_ood.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
from toxicity_screening.pipeline import calibrate_qsar_models
summary=calibrate_qsar_models(ROOT); display(summary)

,endpoint,selected_model,selected_calibrator,validation_pr_auc,validation_brier,test_n,test_positive_prevalence,test_threshold,test_roc_auc,test_pr_auc,...,test_f1,test_brier,test_ece,test_nll,test_recall_at_precision_0.80,test_precision_at_recall_0.80,test_tn,test_fp,test_fn,test_tp
0,herg_blockade,svm,isotonic,0.863559,0.155789,1883,0.507169,0.706667,0.825453,0.820808,...,0.703400,0.172086,0.051978,0.516332,0.509948,0.704505,777,151,355,600
1,ames_mutagenicity,svm,isotonic,0.878349,0.152298,1120,0.547321,0.739130,0.863579,0.879129,...,0.758303,0.151705,0.038207,0.474945,0.766721,0.799358,447,60,202,411
2,SR-p53,xgboost,isotonic,0.313281,0.059326,986,0.085193,0.285714,0.780811,0.322537,...,0.378788,0.066407,0.010840,0.251084,0.000000,0.128571,879,23,59,25
3,SR-ATAD5,svm,isotonic,0.227715,0.035375,1027,0.064265,0.230769,0.771040,0.186695,...,0.280702,0.055984,0.028643,0.234932,0.015152,0.108108,929,32,50,16
4,SR-ARE,svm,isotonic,0.357304,0.118712,847,0.205431,0.290323,0.807207,0.503918,...,0.545455,0.133892,0.050897,0.449368,0.063218,0.362745,551,122,63,111
5,SR-MMP,svm,isotonic,0.614095,0.086364,842,0.220903,0.439252,0.875979,0.643461,...,0.644068,0.112932,0.037625,0.385589,0.311828,0.496914,562,94,53,133


In [3]:
from pathlib import Path
import os
import subprocess
import textwrap

# Use the stable base-Anaconda Python for plotting, not the toxicity kernel.
plot_python = Path(
    os.environ.get(
        "TOXICITY_PLOT_PYTHON",
        r"D:\Users\anaconda3\python.exe",
    )
)

if not plot_python.exists():
    raise FileNotFoundError(
        f"Stable plotting Python was not found: {plot_python}"
    )

worker_path = ROOT / "reports" / "_notebook16_reliability_worker.py"

worker_code = textwrap.dedent(
    r'''
    from pathlib import Path
    import sys

    import numpy as np
    import pandas as pd

    import matplotlib
    matplotlib.use("Agg", force=True)

    import matplotlib.pyplot as plt


    root = Path(sys.argv[1]).resolve()

    summary_path = (
        root
        / "results"
        / "calibration"
        / "calibrated_model_summary.csv"
    )

    if not summary_path.exists():
        raise FileNotFoundError(
            f"Calibration summary does not exist: {summary_path}"
        )

    summary = pd.read_csv(summary_path)

    required_summary_columns = {
        "endpoint",
        "selected_model",
    }

    missing_summary_columns = (
        required_summary_columns - set(summary.columns)
    )

    if missing_summary_columns:
        raise ValueError(
            "Calibration summary is missing columns: "
            f"{sorted(missing_summary_columns)}"
        )

    figure_directory = root / "figures"
    figure_directory.mkdir(parents=True, exist_ok=True)

    created = []
    skipped = []

    selected = (
        summary[
            [
                "endpoint",
                "selected_model",
            ]
        ]
        .dropna()
        .drop_duplicates()
    )

    for row in selected.itertuples(index=False):
        endpoint = str(row.endpoint)
        model_name = str(row.selected_model)

        prediction_path = (
            root
            / "results"
            / "predictions"
            / f"{endpoint}_{model_name}_test.csv"
        )

        if not prediction_path.exists():
            skipped.append(
                {
                    "endpoint": endpoint,
                    "reason": "prediction_file_missing",
                    "path": str(prediction_path),
                }
            )

            print(
                f"[PLOT] skipped endpoint={endpoint} "
                f"reason=prediction_file_missing",
                flush=True,
            )
            continue

        prediction = pd.read_csv(prediction_path)

        required_prediction_columns = {
            "label",
            "probability",
        }

        missing_prediction_columns = (
            required_prediction_columns
            - set(prediction.columns)
        )

        if missing_prediction_columns:
            raise ValueError(
                f"{prediction_path} is missing columns: "
                f"{sorted(missing_prediction_columns)}"
            )

        labels = pd.to_numeric(
            prediction["label"],
            errors="coerce",
        )

        probabilities = pd.to_numeric(
            prediction["probability"],
            errors="coerce",
        )

        valid = labels.notna() & probabilities.notna()

        labels = (
            labels.loc[valid]
            .astype(int)
            .to_numpy()
        )

        probabilities = np.clip(
            probabilities.loc[valid].to_numpy(dtype=float),
            0.0,
            1.0,
        )

        if len(labels) == 0:
            skipped.append(
                {
                    "endpoint": endpoint,
                    "reason": "no_valid_predictions",
                    "path": str(prediction_path),
                }
            )
            continue

        # Quantile-style reliability bins without importing sklearn.
        order = np.argsort(probabilities)

        number_of_bins = min(
            10,
            len(probabilities),
        )

        index_bins = np.array_split(
            order,
            number_of_bins,
        )

        mean_probability = []
        observed_fraction = []

        for indices in index_bins:
            if len(indices) == 0:
                continue

            mean_probability.append(
                float(
                    probabilities[indices].mean()
                )
            )

            observed_fraction.append(
                float(
                    labels[indices].mean()
                )
            )

        figure, axis = plt.subplots(
            figsize=(6, 5)
        )

        axis.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Ideal",
        )

        axis.plot(
            mean_probability,
            observed_fraction,
            marker="o",
            label="Model",
        )

        axis.set(
            xlabel="Mean predicted probability",
            ylabel="Observed positive fraction",
            title=f"{endpoint}: raw reliability",
            xlim=(0, 1),
            ylim=(0, 1),
        )

        axis.legend()
        figure.tight_layout()

        output_path = (
            figure_directory
            / f"reliability_{endpoint}.png"
        )

        figure.savefig(
            output_path,
            dpi=200,
        )

        plt.close(figure)

        created.append(
            {
                "endpoint": endpoint,
                "model": model_name,
                "output": str(output_path),
                "records": int(len(labels)),
            }
        )

        print(
            f"[PLOT] created endpoint={endpoint} "
            f"model={model_name} "
            f"records={len(labels)} "
            f"output={output_path}",
            flush=True,
        )

    report = pd.DataFrame(created)
    report_path = (
        root
        / "results"
        / "calibration"
        / "reliability_plot_status.csv"
    )

    report.to_csv(
        report_path,
        index=False,
    )

    print(
        f"[PLOT] completed "
        f"created={len(created)} "
        f"skipped={len(skipped)} "
        f"report={report_path}",
        flush=True,
    )
    '''
)

worker_path.write_text(
    worker_code,
    encoding="utf-8",
)

plot_environment = os.environ.copy()

plot_environment.update(
    {
        "MPLBACKEND": "Agg",
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "NUMEXPR_NUM_THREADS": "1",
        "PYTHONUNBUFFERED": "1",
        "PYTHONFAULTHANDLER": "1",
        "MPLCONFIGDIR": str(
            ROOT / "reports" / "_matplotlib_config"
        ),
    }
)

result = subprocess.run(
    [
        str(plot_python),
        "-X",
        "faulthandler",
        str(worker_path),
        str(ROOT),
    ],
    cwd=str(ROOT),
    env=plot_environment,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

    raise RuntimeError(
        "Reliability plotting failed in the external "
        f"plotting process with exit code {result.returncode}"
    )

expected_figures = sorted(
    (ROOT / "figures").glob(
        "reliability_*.png"
    )
)

print(
    {
        "plot_python": str(plot_python),
        "figures_created": len(expected_figures),
        "figures": [
            path.name
            for path in expected_figures
        ],
    }
)

[PLOT] created endpoint=herg_blockade model=svm records=1883 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\reliability_herg_blockade.png
[PLOT] created endpoint=ames_mutagenicity model=svm records=1120 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\reliability_ames_mutagenicity.png
[PLOT] created endpoint=SR-p53 model=xgboost records=986 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\reliability_SR-p53.png
[PLOT] created endpoint=SR-ATAD5 model=svm records=1027 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\reliability_SR-ATAD5.png
[PLOT] created endpoint=SR-ARE model=svm records=847 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\reliability_SR-ARE.png
[PLOT] created endpoint=SR-MMP model=svm records=842 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\reliability_SR-MMP.png
[PLOT] completed created=6 skipped=0 report=D:\Dropbo

In [4]:
# Split-conformal prediction sets fitted on validation;
# empirical coverage is evaluated on the held-out test partition.

import joblib
import numpy as np
import pandas as pd

from toxicity_screening.evaluation import predict_qsar_probability
from toxicity_screening.uncertainty import (
    SplitConformalBinaryClassifier,
)

records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
)

archive = np.load(
    ROOT / "data/processed/morgan_features.npz",
    allow_pickle=False,
)

X = archive["X"]
ids = archive["molecule_id"].astype(str)

if X.shape[0] != len(ids):
    raise ValueError(
        f"Feature/identifier mismatch: "
        f"{X.shape[0]} feature rows versus {len(ids)} IDs"
    )

feature_index = pd.DataFrame(
    {
        "molecule_id": ids,
        "row": np.arange(
            len(ids),
            dtype=np.int64,
        ),
    }
)

records = records.merge(
    feature_index,
    on="molecule_id",
    validate="many_to_one",
)

rows = []

for endpoint in summary["endpoint"].astype(str):
    summary_row = summary.loc[
        summary["endpoint"].astype(str) == endpoint
    ].iloc[0]

    selected_model = str(
        summary_row["selected_model"]
    )

    bundle_path = (
        ROOT
        / "models"
        / "calibrated"
        / f"{endpoint}.joblib"
    )

    if not bundle_path.exists():
        raise FileNotFoundError(
            f"Calibrated bundle does not exist: {bundle_path}"
        )

    bundle = joblib.load(bundle_path)

    frame = records.loc[
        (records["endpoint"] == endpoint)
        & records["label"].notna()
    ].copy()

    validation = frame.loc[
        frame["scaffold_split"] == "validation"
    ].copy()

    test = frame.loc[
        frame["scaffold_split"] == "test"
    ].copy()

    if validation.empty or test.empty:
        raise ValueError(
            f"{endpoint}: validation or test partition is empty"
        )

    y_validation = (
        validation["label"]
        .astype(int)
        .to_numpy()
    )

    y_test = (
        test["label"]
        .astype(int)
        .to_numpy()
    )

    x_validation = X[
        validation["row"].astype(int).to_numpy()
    ]

    x_test = X[
        test["row"].astype(int).to_numpy()
    ]

    raw_validation = predict_qsar_probability(
        bundle.model,
        x_validation,
        model_name=selected_model,
    )

    raw_test = predict_qsar_probability(
        bundle.model,
        x_test,
        model_name=selected_model,
    )

    calibrated_validation = (
        bundle.calibrator.predict(raw_validation)
        if bundle.calibrator is not None
        else raw_validation
    )

    calibrated_test = (
        bundle.calibrator.predict(raw_test)
        if bundle.calibrator is not None
        else raw_test
    )

    conformal = SplitConformalBinaryClassifier(
        alpha=0.1
    ).fit(
        calibrated_validation,
        y_validation,
    )

    test_coverage = conformal.coverage(
        calibrated_test,
        y_test,
    )

    mean_set_size = conformal.mean_set_size(
        calibrated_test
    )

    rows.append(
        {
            "endpoint": endpoint,
            "selected_model": selected_model,
            "alpha": 0.1,
            "validation_n": int(len(validation)),
            "test_n": int(len(test)),
            "test_coverage": float(test_coverage),
            "mean_prediction_set_size": float(
                mean_set_size
            ),
        }
    )

    print(
        f"[CONFORMAL] endpoint={endpoint} "
        f"model={selected_model} "
        f"test_coverage={test_coverage:.4f} "
        f"mean_set_size={mean_set_size:.4f}",
        flush=True,
    )

conformal_summary = pd.DataFrame(rows)

output_path = (
    ROOT
    / "results"
    / "uncertainty"
    / "conformal_summary.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

conformal_summary.to_csv(
    output_path,
    index=False,
)

print(
    f"[CONFORMAL] completed "
    f"endpoints={len(conformal_summary)} "
    f"output={output_path}",
    flush=True,
)

display(conformal_summary)

[CONFORMAL] endpoint=herg_blockade model=svm test_coverage=0.9315 mean_set_size=1.4668
[CONFORMAL] endpoint=ames_mutagenicity model=svm test_coverage=0.9330 mean_set_size=1.3929
[CONFORMAL] endpoint=SR-p53 model=xgboost test_coverage=0.9391 mean_set_size=1.7353
[CONFORMAL] endpoint=SR-ATAD5 model=svm test_coverage=0.9649 mean_set_size=1.7274
[CONFORMAL] endpoint=SR-ARE model=svm test_coverage=0.8406 mean_set_size=1.4215
[CONFORMAL] endpoint=SR-MMP model=svm test_coverage=0.8717 mean_set_size=1.2613
[CONFORMAL] completed endpoints=6 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\results\uncertainty\conformal_summary.csv


,endpoint,selected_model,alpha,validation_n,test_n,test_coverage,mean_prediction_set_size
0,herg_blockade,svm,0.1,2095,1883,0.931492,1.466808
1,ames_mutagenicity,svm,0.1,1111,1120,0.933036,1.392857
2,SR-p53,xgboost,0.1,786,986,0.939148,1.735294
3,SR-ATAD5,svm,0.1,829,1027,0.964946,1.727361
4,SR-ARE,svm,0.1,680,847,0.840614,1.421488
5,SR-MMP,svm,0.1,677,842,0.871734,1.261283


### Completion gate
Confirm that the declared artifacts exist before continuing to `17_applicability_domain_and_ood.ipynb`.